# MarginGraph retail store simulation

This notebook runs MarginGraph inside a simulated store, one day at a time.

**Flow for each checkout:** a shopper arrives with a cart, the candidate pool is built, LightGBM scores p(accept) for the top 20 items, the policy picks up to 2 add-ons, the shopper accepts or rejects, and then stock and profit update.

- Shoppers and carts are **real** Online Retail II test checkouts (Oct to Dec 2011), replayed in time order and split into 30 days.
- Probabilities come from the **real** trained LightGBM ranker.
- Cost, stock and overstock are **simulated**, because the dataset has none. Stock is refilled every 7 days.
- *Replay* means the shopper accepts an add-on only if it is the item they really bought. *Stochastic* means they accept with the calibrated p (5 seeds).

Prerequisite (about 10 minutes on CPU): `margingraph prepare && margingraph run --results ART/results --artifacts ART/artifacts`

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt
from margingraph.sim import POLICIES, run_store
ART = os.environ.get('SIM_ART', '/tmp/claude-0/sim')
slates = pd.read_parquet(f'{ART}/artifacts/test_slates.parquet')
erp = pd.read_csv(f'{ART}/results/erp_simulated.csv')
print(slates['event_id'].nunique(), 'checkouts;', len(erp), 'products;', int(erp['overstock'].sum()), 'overstocked')
print('simulated margin (median):', round(float(((erp.price-erp.unit_cost)/erp.price).median()), 3))
slates.head()

## One checkout, step by step

In [ ]:
from margingraph.optimize import solve_enum
g = slates[slates.event_id == slates.event_id.iloc[0]].reset_index(drop=True)
print('cart:', g['context'].iloc[0][:200])
print('really bought (hidden):', g['target'].iloc[0])
items = g.item_idx.to_numpy(); pr, co = erp.price.to_numpy()[items], erp.unit_cost.to_numpy()[items]
show = g[['item','description','p']].assign(price=pr, cost=co, margin=((pr-co)/pr).round(3), overstock=erp.overstock.to_numpy()[items])
print(show.head(10).to_string())
for name,(kind,pol) in POLICIES.items():
    if kind=='topk': ch=[(0,0.0),(1,0.0)]
    else: ch = solve_enum(g.p.to_numpy(), pr, co, erp.overstock.to_numpy()[items], erp.stock.to_numpy()[items], pol, float((erp.price-erp.unit_cost).clip(lower=0).mean()))
    print(f'{name:26s} ->', [g.description[j] for j,_ in ch])

## Run the store for 30 days for every policy

In [ ]:
runs = []
for pol in POLICIES:
    runs.append(run_store(slates, erp, pol, days=30, seed=None))
    for s in range(5):
        runs.append(run_store(slates, erp, pol, days=30, seed=s))
daily = pd.concat(runs, ignore_index=True)
tot = daily.groupby(['policy','seed']).sum(numeric_only=True).drop(columns='day').reset_index()
tot['mode'] = tot.seed.map(lambda s: 'replay' if s < 0 else 'stochastic')
summ = tot.groupby(['mode','policy'])[['offers','accepts','revenue','cost','profit','overstock_units','stockout_blocks','floor_breaks']].mean()
summ['accept_rate'] = summ.accepts/summ.offers; summ['margin'] = summ.profit/summ.revenue
summ['profit_vs_ranker_%'] = 100*(summ.profit/summ.groupby(level=0).profit.transform(lambda x: x.xs('ranker_top2', level=1).iloc[0]) - 1)
summ.round(3)

## Cumulative profit over the 30 days (replay acceptance)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i,(mode,sub) in enumerate([('replay', daily[daily.seed<0]), ('stochastic (mean of 5 seeds)', daily[daily.seed>=0])]):
    c = sub.groupby(['policy','day']).profit.mean().groupby(level=0).cumsum().unstack(0)
    c.plot(ax=ax[i]); ax[i].set_title(mode); ax[i].set_xlabel('simulated day'); ax[i].set_ylabel('cumulative add-on profit (GBP)')
plt.tight_layout(); plt.show()

In [ ]:
per_day = daily[daily.seed>=0].groupby(['policy','day'])[['accepts','overstock_units','stockout_blocks']].mean().groupby(level=0).sum()
per_day